In [0]:
%sql
CREATE OR REPLACE TABLE retail.gold.dim_calendar AS
WITH date_sequence AS (
  SELECT explode(sequence(DATE '2025-01-01', DATE '2027-12-31', INTERVAL 1 DAY)) AS calendar_date
)
SELECT
  -- Date Key & Date
  CAST(date_format(calendar_date, 'yyyyMMdd') AS INT) AS date_key,
  calendar_date,

  -- Year
  year(calendar_date) AS year,
  
  -- Quarter
  quarter(calendar_date) AS quarter,
  CONCAT('Q', quarter(calendar_date)) AS quarter_name,
  CONCAT(year(calendar_date), '-Q', quarter(calendar_date)) AS year_quarter,

  -- Month
  month(calendar_date) AS month_number,
  date_format(calendar_date, 'MMMM') AS month_name,
  date_format(calendar_date, 'MMM') AS month_short_name,
  CONCAT(year(calendar_date), '-', LPAD(month(calendar_date), 2, '0')) AS year_month,

  -- Week
  weekofyear(calendar_date) AS week_of_year,
  CONCAT(year(calendar_date), '-W', LPAD(weekofyear(calendar_date), 2, '0')) AS year_week,

  -- Day
  day(calendar_date) AS day_of_month,
  dayofweek(calendar_date) AS day_of_week,
  dayofyear(calendar_date) AS day_of_year,
  date_format(calendar_date, 'EEEE') AS day_name,
  date_format(calendar_date, 'EEE') AS day_short_name,

  -- Week Flags
  CASE WHEN dayofweek(calendar_date) IN (1, 7) THEN TRUE ELSE FALSE END AS is_weekend,
  CASE WHEN dayofweek(calendar_date) NOT IN (1, 7) THEN TRUE ELSE FALSE END AS is_weekday,

  -- Month Boundaries
  trunc(calendar_date, 'MM') AS first_day_of_month,
  last_day(calendar_date) AS last_day_of_month,
  CASE WHEN calendar_date = last_day(calendar_date) THEN TRUE ELSE FALSE END AS is_last_day_of_month,

  -- Quarter Boundaries
  trunc(calendar_date, 'QQ') AS first_day_of_quarter,

  -- Year Boundaries
  trunc(calendar_date, 'YY') AS first_day_of_year,

  -- Fiscal Year (assuming Apr-Mar fiscal year)
  CASE WHEN month(calendar_date) >= 4 THEN year(calendar_date) ELSE year(calendar_date) - 1 END AS fiscal_year,
  CASE 
    WHEN month(calendar_date) >= 4 THEN quarter(calendar_date) - 1
    ELSE quarter(calendar_date) + 3
  END AS fiscal_quarter,

  -- Days in Month
  day(last_day(calendar_date)) AS days_in_month

FROM date_sequence
ORDER BY calendar_date